# LLMs Encode Harmfulness and Refusal Separately
## Complete Experiment Notebook

This notebook provides a complete, standalone implementation of all experiments from the paper:
**"LLMs Encode Harmfulness and Refusal Separately"**

### Key Findings:
- **t_inst** (last token of instruction): encodes **harmfulness**
- **t_post-inst** (last token of prompt): encodes **refusal behavior**

### Experiments:
1. **Hidden State Extraction** - Extract activations at t_inst and t_post-inst
2. **Direction Computation** - Compute harmfulness and refusal directions
3. **LAT Probe Training** - Train linear classifier for harmfulness detection
4. **Piece-wise Intervention** - Conditionally suppress refusal for benign prompts
5. **Evaluation** - Measure safety/compliance tradeoff

---

## 0. Setup and Imports

In [ ]:
# Standard imports
import os
import sys
import json
import functools
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional, Callable

import numpy as np
import torch
import torch.nn as nn
from torch import Tensor
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve

from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

# Set up paths
PROJECT_ROOT = Path("../").resolve()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Add source directories to path
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(PROJECT_ROOT / "extraction"))
sys.path.insert(0, str(PROJECT_ROOT / "steering"))
sys.path.insert(0, str(PROJECT_ROOT / "evaluation"))

# Import from original src
from utils import read_row, formatInp_llama_persuasion, REFUSAL_PHRASE, store_row
from template_inversion import inversion_prompts_choice

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Project root: {PROJECT_ROOT}")

## 1. Configuration

In [ ]:
# ============== CONFIGURATION ==============

CONFIG = {
    # Model settings
    "model_type": "qwen",  # Options: "llama2", "llama3", "qwen"
    "model_paths": {
        "llama2": "NousResearch/Llama-2-7b-chat-hf",
        "llama3": "meta-llama/Meta-Llama-3-8B-Instruct",
        "qwen": "Qwen/Qwen2-7B-Instruct",
    },
    
    # Data paths (relative to DATA_DIR)
    "harmful_file": "advbench.json",
    "harmless_file": "xstest-harmless.json",
    "jailbreak_file": "sorry-badq.json",  # Accepted harmful (jailbroken)
    
    # Experiment settings
    "num_samples": 100,  # Number of samples per category
    "batch_size": 1,     # Keep at 1 for memory efficiency
    "max_new_tokens": 128,
    
    # Layer settings (will be auto-computed based on model)
    "l_lat": None,       # Layer for harmfulness detection (default: num_layers // 2)
    "l_post": None,      # Layer for refusal intervention (default: num_layers - 1)
    
    # Intervention settings
    "alpha": 1.0,        # Steering coefficient
    "tau": 0.0,          # Harmfulness threshold (will be optimized)
    
    # Ridge regression
    "lambda_reg": 1e-3,
}

# Get model path
MODEL_PATH = CONFIG["model_paths"][CONFIG["model_type"]]
print(f"Model: {MODEL_PATH}")
print(f"Model type: {CONFIG['model_type']}")

## 2. Load Model and Tokenizer

In [ ]:
def load_model_and_tokenizer(model_path: str, model_type: str):
    """
    Load model and tokenizer with appropriate settings.
    """
    print(f"Loading model: {model_path}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_path, 
        trust_remote_code=True,
        cache_dir="models/"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        cache_dir="models/"
    )
    model.eval()
    
    # Get number of layers
    num_layers = model.config.num_hidden_layers
    print(f"Model loaded: {num_layers} layers, hidden_size={model.config.hidden_size}")
    
    return model, tokenizer, num_layers

# Load model
model, tokenizer, NUM_LAYERS = load_model_and_tokenizer(MODEL_PATH, CONFIG["model_type"])

# Set layer indices
CONFIG["l_lat"] = NUM_LAYERS // 2  # Middle layer for harmfulness
CONFIG["l_post"] = NUM_LAYERS - 1  # Last layer for refusal
print(f"L_lat (harmfulness): {CONFIG['l_lat']}")
print(f"L_post (refusal): {CONFIG['l_post']}")

## 3. Load and Prepare Data

In [ ]:
def load_prompts(file_path: Path, limit: int = None, key: str = None) -> List[str]:
    """
    Load prompts from JSON/JSONL file.
    Handles various formats from the original repo.
    """
    data = read_row(str(file_path))
    prompts = []
    
    for item in data:
        if isinstance(item, dict):
            # Try different keys based on dataset format
            if key and key in item:
                prompts.append(item[key])
            elif 'instruction' in item:
                prompts.append(item['instruction'])
            elif 'question' in item:
                prompts.append(item['question'])
            elif 'bad_q' in item:
                prompts.append(item['bad_q'])
            elif 'prompt' in item:
                prompts.append(item['prompt'])
        elif isinstance(item, str):
            prompts.append(item)
    
    if limit:
        prompts = prompts[:limit]
    
    return prompts

# Load datasets
print("Loading datasets...")
harmful_prompts = load_prompts(DATA_DIR / CONFIG["harmful_file"], CONFIG["num_samples"])
harmless_prompts = load_prompts(DATA_DIR / CONFIG["harmless_file"], CONFIG["num_samples"])

print(f"Harmful prompts: {len(harmful_prompts)}")
print(f"Harmless prompts: {len(harmless_prompts)}")

# Show examples
print("\n--- Example Harmful Prompt ---")
print(harmful_prompts[0][:200])
print("\n--- Example Harmless Prompt ---")
print(harmless_prompts[0][:200])

## 4. Hidden State Extraction

Extract activations at:
- **t_inst**: Last token of instruction (encodes harmfulness)
- **t_post-inst**: Last token of formatted prompt (encodes refusal)

In [ ]:
def get_block_modules(model) -> List[nn.Module]:
    """
    Get transformer layers from different model architectures.
    Matches the original src/extract_hidden.py approach.
    """
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        return list(model.model.layers)  # LLaMA, Qwen
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
        return list(model.transformer.h)  # GPT-2 style
    else:
        raise ValueError("Unknown model architecture")

def extract_activations(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompts: List[str],
    layer: int,
    model_type: str,
    position: int = -1,  # -1 for last token
    use_pre_hook: bool = True,  # Original code uses pre-hook for input activations
) -> Tensor:
    """
    Extract activations at a specific layer and token position.
    
    Based on original src/extract_hidden.py implementation.
    """
    model.eval()
    block_modules = get_block_modules(model)
    activations = []
    activation_storage = [None]
    
    def hook_fn(module, input, output=None):
        # For pre-hook: use input[0]
        # For forward hook: use output[0]
        if use_pre_hook:
            hidden = input[0] if isinstance(input, tuple) else input
        else:
            hidden = output[0] if isinstance(output, tuple) else output
        
        # Extract at position (half precision as in original)
        activation_storage[0] = hidden[:, position, :].half().detach().cpu()
    
    if use_pre_hook:
        handle = block_modules[layer].register_forward_pre_hook(hook_fn)
    else:
        handle = block_modules[layer].register_forward_hook(hook_fn)
    
    try:
        for prompt in tqdm(prompts, desc=f"Extracting L{layer}"):
            # Format prompt using original function
            formatted = formatInp_llama_persuasion(
                {'instruction': prompt} if isinstance(prompt, str) else prompt,
                model=model_type
            )
            
            inputs = tokenizer(
                formatted,
                return_tensors="pt",
                truncation=True,
                max_length=2048
            ).to(model.device)
            
            with torch.no_grad():
                _ = model(**inputs)
            
            if activation_storage[0] is not None:
                activations.append(activation_storage[0])
    finally:
        handle.remove()
    
    return torch.cat(activations, dim=0)  # [N, hidden_dim]

# Test extraction
print("Testing activation extraction...")
test_acts = extract_activations(
    model, tokenizer, harmful_prompts[:2], 
    layer=CONFIG["l_lat"], 
    model_type=CONFIG["model_type"]
)
print(f"Test activation shape: {test_acts.shape}")

In [ ]:
# Extract activations for harmful prompts at L_lat (harmfulness layer)
print("Extracting harmful activations at L_lat...")
harmful_acts_lat = extract_activations(
    model, tokenizer, harmful_prompts,
    layer=CONFIG["l_lat"],
    model_type=CONFIG["model_type"]
)
print(f"Harmful activations shape: {harmful_acts_lat.shape}")

# Extract activations for harmless prompts at L_lat
print("\nExtracting harmless activations at L_lat...")
harmless_acts_lat = extract_activations(
    model, tokenizer, harmless_prompts,
    layer=CONFIG["l_lat"],
    model_type=CONFIG["model_type"]
)
print(f"Harmless activations shape: {harmless_acts_lat.shape}")

# Save activations
torch.save(harmful_acts_lat, OUTPUT_DIR / "harmful_acts_lat.pt")
torch.save(harmless_acts_lat, OUTPUT_DIR / "harmless_acts_lat.pt")
print("\nActivations saved!")

## 5. Compute Direction Vectors

### 5.1 Harmfulness Direction (v_harm)
Using LAT (Latent Adversarial Training) - ridge regression approach

In [ ]:
def fit_lat_probe(
    H_harmful: Tensor,
    H_harmless: Tensor,
    lam: float = 1e-3
) -> Tuple[Tensor, np.ndarray, np.ndarray]:
    """
    Fit LAT probe using ridge regression.
    
    Returns:
        v_harm: Normalized harmfulness direction [hidden_dim]
        scores: Harmfulness scores for all samples
        labels: Ground truth labels (1=harmful, 0=harmless)
    """
    # Combine into feature matrix
    H = np.vstack([H_harmful.numpy(), H_harmless.numpy()]).astype(np.float32)
    labels = np.array([1.0] * len(H_harmful) + [0.0] * len(H_harmless))
    
    # Ridge regression: w = (H^T H + λI)^{-1} H^T y
    D = H.shape[1]
    HtH = H.T @ H
    HtY = H.T @ labels
    
    w = np.linalg.solve(HtH + lam * np.eye(D), HtY)
    
    # Normalize
    w_norm = w / (np.linalg.norm(w) + 1e-8)
    
    # Compute scores
    scores = H @ w_norm
    
    v_harm = torch.tensor(w_norm, dtype=torch.float32)
    
    return v_harm, scores, labels

# Fit LAT probe
print("Fitting LAT probe...")
v_harm, scores, labels = fit_lat_probe(
    harmful_acts_lat,
    harmless_acts_lat,
    lam=CONFIG["lambda_reg"]
)

# Compute AUC
auc = roc_auc_score(labels, scores)
print(f"\nLAT Probe AUC: {auc:.4f}")

# Score statistics
harmful_scores = scores[labels == 1]
harmless_scores = scores[labels == 0]
print(f"\nHarmful scores:  mean={harmful_scores.mean():.4f}, std={harmful_scores.std():.4f}")
print(f"Harmless scores: mean={harmless_scores.mean():.4f}, std={harmless_scores.std():.4f}")

# Save
torch.save(v_harm, OUTPUT_DIR / "v_harm.pt")
print(f"\nSaved v_harm.pt (shape: {v_harm.shape})")

In [ ]:
# Visualize score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(harmful_scores, bins=30, alpha=0.6, label=f'Harmful (n={len(harmful_scores)})', color='red')
ax.hist(harmless_scores, bins=30, alpha=0.6, label=f'Harmless (n={len(harmless_scores)})', color='green')
ax.axvline(harmful_scores.mean(), color='red', linestyle='--', label=f'Harmful mean: {harmful_scores.mean():.3f}')
ax.axvline(harmless_scores.mean(), color='green', linestyle='--', label=f'Harmless mean: {harmless_scores.mean():.3f}')
ax.set_xlabel('Harmfulness Score')
ax.set_ylabel('Count')
ax.set_title('Distribution of LAT Harmfulness Scores')
ax.legend()

# ROC curve
ax = axes[1]
fpr, tpr, thresholds = roc_curve(labels, scores)
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'LAT Probe (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', label='Random')

# Mark optimal point (Youden's J)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
ax.scatter([fpr[best_idx]], [tpr[best_idx]], s=100, c='red', zorder=5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve for Harmfulness Detection')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'lat_analysis.png', dpi=150)
plt.show()

### 5.2 Refusal Direction (v_refuse)

Compute mean difference between refused and accepted responses at t_post-inst

In [ ]:
def detect_refusal(response: str) -> bool:
    """Detect if a response is a refusal using phrases from original utils.py"""
    if not response:
        return False
    response_lower = response.lower()
    for phrase in REFUSAL_PHRASE:
        if phrase.lower() in response_lower:
            return True
    return False

def generate_response(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    model_type: str,
    max_new_tokens: int = 128
) -> str:
    """
    Generate a response for a single prompt.
    """
    formatted = formatInp_llama_persuasion(
        {'instruction': prompt} if isinstance(prompt, str) else prompt,
        model=model_type
    )
    
    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)
    
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    with torch.no_grad():
        outputs = model.generate(**inputs, generation_config=gen_config)
    
    # Extract only generated tokens
    generated = outputs[0, inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    
    return response.strip()

# Test generation
test_response = generate_response(model, tokenizer, harmless_prompts[0], CONFIG["model_type"])
print("Test response:")
print(test_response[:200])

In [ ]:
# Run inference on harmless prompts to get refused/accepted labels
# (Harmless prompts that get refused are over-refusals we want to fix)

print("Running inference to identify refused vs accepted...")
print("(This may take a while)\n")

harmless_results = []
for i, prompt in enumerate(tqdm(harmless_prompts, desc="Harmless")):
    response = generate_response(model, tokenizer, prompt, CONFIG["model_type"])
    refused = detect_refusal(response)
    harmless_results.append({
        'prompt': prompt,
        'response': response,
        'refused': refused,
        'category': 'harmless'
    })

# Count
n_refused = sum(1 for r in harmless_results if r['refused'])
n_accepted = len(harmless_results) - n_refused
print(f"\nHarmless prompts: {n_refused} refused, {n_accepted} accepted")
print(f"Over-refusal rate: {n_refused/len(harmless_results):.1%}")

# Save results
store_row(str(OUTPUT_DIR / "harmless_inference.jsonl"), harmless_results)

In [ ]:
# Extract activations at L_post for refused vs accepted
# We need both harmful (should be refused) and harmless results

print("Running inference on harmful prompts...")
harmful_results = []
for prompt in tqdm(harmful_prompts, desc="Harmful"):
    response = generate_response(model, tokenizer, prompt, CONFIG["model_type"])
    refused = detect_refusal(response)
    harmful_results.append({
        'prompt': prompt,
        'response': response,
        'refused': refused,
        'category': 'harmful'
    })

n_harmful_refused = sum(1 for r in harmful_results if r['refused'])
print(f"\nHarmful prompts: {n_harmful_refused}/{len(harmful_results)} refused")
print(f"Safety rate: {n_harmful_refused/len(harmful_results):.1%}")

store_row(str(OUTPUT_DIR / "harmful_inference.jsonl"), harmful_results)

In [ ]:
# Compute refusal vector: mean_refused - mean_accepted at L_post
# Use all refused vs all accepted prompts

refused_prompts = [r['prompt'] for r in harmless_results + harmful_results if r['refused']]
accepted_prompts = [r['prompt'] for r in harmless_results + harmful_results if not r['refused']]

print(f"Refused prompts: {len(refused_prompts)}")
print(f"Accepted prompts: {len(accepted_prompts)}")

if len(refused_prompts) > 0 and len(accepted_prompts) > 0:
    # Extract at L_post
    print("\nExtracting refused activations at L_post...")
    refused_acts = extract_activations(
        model, tokenizer, refused_prompts,
        layer=CONFIG["l_post"],
        model_type=CONFIG["model_type"]
    )
    
    print("Extracting accepted activations at L_post...")
    accepted_acts = extract_activations(
        model, tokenizer, accepted_prompts,
        layer=CONFIG["l_post"],
        model_type=CONFIG["model_type"]
    )
    
    # Compute refusal direction
    mean_refused = refused_acts.mean(dim=0)
    mean_accepted = accepted_acts.mean(dim=0)
    
    v_refuse = mean_refused - mean_accepted
    v_refuse = v_refuse / (torch.norm(v_refuse) + 1e-8)  # Normalize
    
    print(f"\nv_refuse shape: {v_refuse.shape}")
    print(f"Vector norm before normalization: {torch.norm(mean_refused - mean_accepted).item():.4f}")
    
    # Save
    torch.save(v_refuse, OUTPUT_DIR / "v_refuse.pt")
    print("Saved v_refuse.pt")
else:
    print("WARNING: Not enough refused/accepted samples to compute v_refuse")
    print("Using mean difference approximation instead...")
    
    # Fallback: use harmful vs harmless at L_post
    v_refuse = (harmful_acts_lat.mean(dim=0) - harmless_acts_lat.mean(dim=0))
    v_refuse = v_refuse / (torch.norm(v_refuse) + 1e-8)
    torch.save(v_refuse, OUTPUT_DIR / "v_refuse.pt")

## 6. Choose Optimal Threshold (τ)

In [ ]:
def compute_metrics_at_threshold(scores, labels, tau):
    """Compute metrics at a given threshold."""
    predictions = (scores > tau).astype(int)
    
    n_harmful = labels.sum()
    n_harmless = len(labels) - n_harmful
    
    tp = ((predictions == 1) & (labels == 1)).sum()
    tn = ((predictions == 0) & (labels == 0)).sum()
    
    safety_rate = tp / n_harmful if n_harmful > 0 else 0  # TPR
    compliance_rate = tn / n_harmless if n_harmless > 0 else 0  # TNR
    tradeoff = (safety_rate + compliance_rate) / 2
    
    return {
        'tau': tau,
        'safety_rate': safety_rate,
        'compliance_rate': compliance_rate,
        'tradeoff': tradeoff
    }

# Find optimal tau using ROC curve
fpr, tpr, thresholds = roc_curve(labels, scores)

# Try different threshold selection methods
tau_candidates = []

# 1. Youden's J statistic (maximize TPR - FPR)
j_scores = tpr - fpr
best_j_idx = np.argmax(j_scores)
tau_youden = thresholds[best_j_idx]
tau_candidates.append(('youden', tau_youden, compute_metrics_at_threshold(scores, labels, tau_youden)))

# 2. High safety constraint (95% safety rate)
valid_idx = np.where(tpr >= 0.95)[0]
if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmin(fpr[valid_idx])]
    tau_safe = thresholds[best_idx]
    tau_candidates.append(('safe_95', tau_safe, compute_metrics_at_threshold(scores, labels, tau_safe)))

# 3. Percentile-based on harmful scores
tau_p10 = np.percentile(harmful_scores, 10)
tau_candidates.append(('harmful_p10', tau_p10, compute_metrics_at_threshold(scores, labels, tau_p10)))

# Print all candidates
print("Threshold Candidates:")
print("-" * 70)
print(f"{'Method':<15} {'Tau':>10} {'Safety':>10} {'Comply':>10} {'Trade':>10}")
print("-" * 70)
for name, tau, metrics in tau_candidates:
    print(f"{name:<15} {tau:>10.4f} {metrics['safety_rate']:>9.1%} {metrics['compliance_rate']:>9.1%} {metrics['tradeoff']:>9.1%}")

# Select best by tradeoff
best_tau = max(tau_candidates, key=lambda x: x[2]['tradeoff'])
print(f"\nSelected: {best_tau[0]} with τ = {best_tau[1]:.4f}")
CONFIG['tau'] = best_tau[1]

## 7. Piece-wise Steering Implementation

The key insight: if `s(x) <= τ` (benign prompt), subtract `α * v_refuse` to suppress refusal.

In [ ]:
class PiecewiseSteeringHooks:
    """
    Implements piece-wise steering based on harmfulness score.
    
    Based on original src/intervention.py but with conditional logic.
    """
    
    def __init__(
        self,
        v_harm: Tensor,
        v_refuse: Tensor,
        tau: float,
        alpha: float,
        l_lat: int,
        l_post: int,
    ):
        self.v_harm = v_harm.float()
        self.v_refuse = v_refuse.float()
        self.tau = tau
        self.alpha = alpha
        self.l_lat = l_lat
        self.l_post = l_post
        
        self.handles = []
        self.score = None  # Store score for inspection
        self.intervened = False
    
    def register(self, model):
        """Register hooks on the model."""
        blocks = get_block_modules(model)
        
        # Hook to read harmfulness score at L_lat
        def read_hook(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            h = hidden[:, -1, :].float()  # Last token
            self.score = torch.matmul(h, self.v_harm.to(h.device)).item()
        
        # Hook to apply steering at L_post
        def steer_hook(module, input, output):
            if self.score is not None and self.score <= self.tau:
                self.intervened = True
                hidden = output[0].clone()
                v = self.v_refuse.to(hidden.device).view(1, 1, -1)
                hidden[:, -1:, :] = hidden[:, -1:, :] - self.alpha * v
                return (hidden,) + output[1:] if len(output) > 1 else (hidden,)
            self.intervened = False
            return output
        
        self.handles.append(blocks[self.l_lat].register_forward_hook(read_hook))
        self.handles.append(blocks[self.l_post].register_forward_hook(steer_hook))
    
    def remove(self):
        """Remove all hooks."""
        for h in self.handles:
            h.remove()
        self.handles = []
    
    def __enter__(self):
        return self
    
    def __exit__(self, *args):
        self.remove()

print("PiecewiseSteeringHooks defined!")

## 8. Run Evaluation: Baseline vs Steered

In [ ]:
def run_evaluation(
    model,
    tokenizer,
    test_prompts: List[Dict],
    model_type: str,
    hooks=None
) -> List[Dict]:
    """
    Run evaluation with optional steering hooks.
    
    Args:
        test_prompts: List of dicts with 'prompt' and 'category' keys
        hooks: Optional PiecewiseSteeringHooks instance
    
    Returns:
        List of results with responses and refusal status
    """
    results = []
    
    if hooks:
        hooks.register(model)
    
    try:
        for item in tqdm(test_prompts, desc="Evaluating"):
            prompt = item['prompt']
            response = generate_response(model, tokenizer, prompt, model_type)
            refused = detect_refusal(response)
            
            result = {
                'prompt': prompt,
                'category': item['category'],
                'response': response,
                'refused': refused,
            }
            
            if hooks:
                result['score'] = hooks.score
                result['intervened'] = hooks.intervened
            
            results.append(result)
    finally:
        if hooks:
            hooks.remove()
    
    return results

def compute_eval_metrics(results):
    """Compute safety and compliance metrics."""
    harmful = [r for r in results if r['category'] == 'harmful']
    harmless = [r for r in results if r['category'] == 'harmless']
    
    safety_rate = sum(1 for r in harmful if r['refused']) / len(harmful) if harmful else 0
    compliance_rate = sum(1 for r in harmless if not r['refused']) / len(harmless) if harmless else 0
    tradeoff = (safety_rate + compliance_rate) / 2
    
    return {
        'safety_rate': safety_rate,
        'compliance_rate': compliance_rate,
        'tradeoff': tradeoff,
        'n_harmful': len(harmful),
        'n_harmless': len(harmless),
    }

In [ ]:
# Prepare test set
test_size = min(30, len(harmful_prompts), len(harmless_prompts))  # Use subset for speed

test_set = (
    [{'prompt': p, 'category': 'harmful'} for p in harmful_prompts[:test_size]] +
    [{'prompt': p, 'category': 'harmless'} for p in harmless_prompts[:test_size]]
)

print(f"Test set: {len(test_set)} prompts ({test_size} harmful + {test_size} harmless)")

# Run baseline (no intervention)
print("\n=== Running Baseline (No Intervention) ===")
baseline_results = run_evaluation(model, tokenizer, test_set, CONFIG["model_type"])
baseline_metrics = compute_eval_metrics(baseline_results)

print(f"\nBaseline Results:")
print(f"  Safety Rate:     {baseline_metrics['safety_rate']:.1%}")
print(f"  Compliance Rate: {baseline_metrics['compliance_rate']:.1%}")
print(f"  Tradeoff Score:  {baseline_metrics['tradeoff']:.1%}")

In [ ]:
# Run with piece-wise steering
print("\n=== Running With Piece-wise Steering ===")
print(f"Parameters: τ={CONFIG['tau']:.4f}, α={CONFIG['alpha']}")

# Load vectors
v_harm = torch.load(OUTPUT_DIR / "v_harm.pt")
v_refuse = torch.load(OUTPUT_DIR / "v_refuse.pt")

# Create hooks
steered_hooks = PiecewiseSteeringHooks(
    v_harm=v_harm,
    v_refuse=v_refuse,
    tau=CONFIG['tau'],
    alpha=CONFIG['alpha'],
    l_lat=CONFIG['l_lat'],
    l_post=CONFIG['l_post'],
)

# Run evaluation
steered_results = run_evaluation(model, tokenizer, test_set, CONFIG["model_type"], hooks=steered_hooks)
steered_metrics = compute_eval_metrics(steered_results)

print(f"\nSteered Results:")
print(f"  Safety Rate:     {steered_metrics['safety_rate']:.1%}")
print(f"  Compliance Rate: {steered_metrics['compliance_rate']:.1%}")
print(f"  Tradeoff Score:  {steered_metrics['tradeoff']:.1%}")

In [ ]:
# Compare results
print("\n" + "="*60)
print("COMPARISON: BASELINE vs STEERED")
print("="*60)
print(f"\n{'Metric':<20} {'Baseline':>12} {'Steered':>12} {'Change':>12}")
print("-"*60)

for metric in ['safety_rate', 'compliance_rate', 'tradeoff']:
    base = baseline_metrics[metric]
    steer = steered_metrics[metric]
    change = steer - base
    print(f"{metric:<20} {base:>11.1%} {steer:>11.1%} {change:>+11.1%}")

print("="*60)

# Show examples where behavior changed
print("\nExamples where behavior CHANGED:")
print("-"*60)

for i in range(len(baseline_results)):
    base = baseline_results[i]
    steer = steered_results[i]
    
    if base['refused'] != steer['refused']:
        print(f"\n[{base['category'].upper()}] {base['prompt'][:80]}...")
        print(f"  Baseline: {'REFUSED' if base['refused'] else 'ACCEPTED'}")
        print(f"  Steered:  {'REFUSED' if steer['refused'] else 'ACCEPTED'} (score={steer.get('score', 0):.4f})")
        if steer.get('intervened'):
            print(f"  → Intervention applied!")

## 9. Parameter Sweep

In [ ]:
# Sweep tau and alpha to find optimal operating point
tau_values = np.linspace(harmful_scores.min(), harmless_scores.max(), 7)
alpha_values = [0.5, 1.0, 1.5, 2.0]

print("Running parameter sweep...")
print(f"Testing {len(tau_values)} tau values x {len(alpha_values)} alpha values\n")

# Use smaller subset for sweep
sweep_size = min(20, len(harmful_prompts), len(harmless_prompts))
sweep_set = (
    [{'prompt': p, 'category': 'harmful'} for p in harmful_prompts[:sweep_size]] +
    [{'prompt': p, 'category': 'harmless'} for p in harmless_prompts[:sweep_size]]
)

sweep_results = []

for tau in tau_values:
    for alpha in alpha_values:
        hooks = PiecewiseSteeringHooks(
            v_harm=v_harm, v_refuse=v_refuse,
            tau=tau, alpha=alpha,
            l_lat=CONFIG['l_lat'], l_post=CONFIG['l_post']
        )
        
        results = run_evaluation(model, tokenizer, sweep_set, CONFIG["model_type"], hooks=hooks)
        metrics = compute_eval_metrics(results)
        
        sweep_results.append({
            'tau': tau,
            'alpha': alpha,
            **metrics
        })
        
        print(f"τ={tau:.3f}, α={alpha:.1f}: Safety={metrics['safety_rate']:.1%}, Comply={metrics['compliance_rate']:.1%}, Trade={metrics['tradeoff']:.1%}")

# Find best
best = max(sweep_results, key=lambda x: x['tradeoff'])
print(f"\nBest: τ={best['tau']:.4f}, α={best['alpha']:.1f} with tradeoff={best['tradeoff']:.1%}")

In [ ]:
# Visualize sweep results
fig, ax = plt.subplots(figsize=(10, 6))

for alpha in alpha_values:
    subset = [r for r in sweep_results if r['alpha'] == alpha]
    taus = [r['tau'] for r in subset]
    tradeoffs = [r['tradeoff'] for r in subset]
    ax.plot(taus, tradeoffs, 'o-', label=f'α={alpha}', linewidth=2, markersize=8)

ax.axhline(baseline_metrics['tradeoff'], color='red', linestyle='--', label='Baseline')
ax.set_xlabel('Tau (τ)')
ax.set_ylabel('Tradeoff Score')
ax.set_title('Parameter Sweep: Tradeoff vs Tau')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'parameter_sweep.png', dpi=150)
plt.show()

## 10. Save All Results

In [ ]:
# Save comprehensive results
final_results = {
    'config': CONFIG,
    'model_path': MODEL_PATH,
    'baseline_metrics': baseline_metrics,
    'steered_metrics': steered_metrics,
    'best_sweep_result': best,
    'all_sweep_results': sweep_results,
    'lat_auc': auc,
    'score_stats': {
        'harmful_mean': float(harmful_scores.mean()),
        'harmful_std': float(harmful_scores.std()),
        'harmless_mean': float(harmless_scores.mean()),
        'harmless_std': float(harmless_scores.std()),
    }
}

with open(OUTPUT_DIR / 'experiment_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("Results saved to outputs/experiment_results.json")
print("\n" + "="*60)
print("EXPERIMENT COMPLETE")
print("="*60)
print(f"\nGenerated files:")
for f in OUTPUT_DIR.glob("*"):
    print(f"  - {f.name}")

---
## Summary

This notebook demonstrated the complete pipeline for:

1. **Hidden State Extraction** at t_inst and t_post-inst positions
2. **LAT Probe Training** to learn harmfulness direction v_harm
3. **Refusal Vector Computation** from refused vs accepted examples
4. **Piece-wise Steering** that conditionally suppresses refusal for benign prompts
5. **Evaluation** comparing baseline vs steered model behavior

Key findings:
- The model encodes harmfulness and refusal as separate concepts
- Steering at t_post-inst can reduce over-refusal without compromising safety
- The optimal threshold τ balances compliance and safety rates